In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [44]:
import tsdm
import torch
from torch import Tensor
from pandas import DataFrame

In [17]:
ds = tsdm.datasets.ETTh1.dataset

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.674,3.550,-5.615,2.132,3.472,1.523,10.904000
2018-06-26 16:00:00,-5.492,4.287,-9.132,2.274,3.533,1.675,11.044000
2018-06-26 17:00:00,2.813,3.818,-0.817,2.097,3.716,1.523,10.271000


In [13]:
?torch.Tensor

Init signature: torch.Tensor(self, /, *args, **kwargs)
Docstring:      <no docstring>
File:           ~/miniconda3/envs/kiwi/lib/python3.9/site-packages/torch/__init__.py
Type:           _TensorMeta
Subclasses:     Parameter, UninitializedBuffer


In [39]:
class TimeTensor(Tensor):
    def __new__(cls, *args, index, **kwargs):
        print("__new__", f"{args=}", f"{index=}", f"{kwargs=}", sep="\n\t")
        return super().__new__(cls, *args, **kwargs)

    def __init__(self, /, *args, index, **kwargs):
        print(
            "__init__", f"{args=}", f"{index=}", f"{kwargs=}", f"{kwargs=}", sep="\n\t"
        )
        super().__init__()
        self.index = index

In [59]:
?ds.index.get_indexer

Signature:
ds.index.get_indexer(
    target,
    method: 'str_t | None' = None,
    limit: 'int | None' = None,
    tolerance=None,
) -> 'np.ndarray'
Docstring:
Compute indexer and mask for new index given the current index. The
indexer should be then used as an input to ndarray.take to align the
current data to the new index.

Parameters
----------
target : Index
method : {None, 'pad'/'ffill', 'backfill'/'bfill', 'nearest'}, optional
    * default: exact matches only.
    * pad / ffill: find the PREVIOUS index value if no exact match.
    * backfill / bfill: use NEXT index value if no exact match
    * nearest: use the NEAREST index value if no exact match. Tied
      distances are broken by preferring the larger index value.
limit : int, optional
    Maximum number of consecutive labels in ``target`` to match for
    inexact matches.
tolerance : optional
    Maximum distance between original and new labels for inexact
    matches. The values of the index at the matching locations mus

In [72]:
ds.index.is_monotonic_increasing

True

In [82]:
from pandas import Timestamp, Period

In [83]:
ds.loc["2016-07-01":"2017-06-30"]
ds.loc[Timestamp("2016-07-01") : Timestamp("2017-06-30")]

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000
...,...,...,...,...,...,...,...
2017-06-29 20:00:00,10.315,0.536,6.325,0.853,3.868,-0.670,20.471001
2017-06-29 21:00:00,10.382,0.737,6.005,0.675,3.838,-0.548,20.752001
2017-06-29 22:00:00,9.377,0.938,5.543,0.711,3.655,-0.487,19.627001


In [87]:
ds.index.get_loc("2016-07-01"), ds.index.get_loc(Timestamp("2016-07-01"))

(slice(0, 24, None), 0)

In [86]:
ds.index.get_loc(Timestamp("2017-06-30")), ds.index.get_loc("2017-06-30")

(8736, slice(8736, 8760, None))

In [66]:
DataFrame(ds.index).reset_index().set_index("date").loc[
    "2016-07-01":"2017-06-30"
].values

,index
date,
2016-07-01 00:00:00,0
2016-07-01 01:00:00,1
2016-07-01 02:00:00,2
2016-07-01 03:00:00,3
2016-07-01 04:00:00,4
...,...
2017-06-30 19:00:00,8755
2017-06-30 20:00:00,8756
2017-06-30 21:00:00,8757


In [52]:
ds.get_loc["2016-07-01":"2017-06-30"]

AttributeError: 'DataFrame' object has no attribute 'get_loc'

In [53]:
idex = DataFrame(ds.index)
idex.get_loc["2016-07-01":"2017-06-30"]

AttributeError: 'DataFrame' object has no attribute 'get_loc'

In [55]:
ds.index.get_loc(["2016-07-01":"2017-06-30"])

SyntaxError: invalid syntax (1585493918.py, line 1)

In [40]:
T = TimeTensor(ds.values, index=ds.index)

__new__
	args=(array([[ 5.82700014,  2.00900006,  1.59899998, ...,  4.20300007,
         1.34000003, 30.53100014],
       [ 5.69299984,  2.07599998,  1.49199998, ...,  4.1420002 ,
         1.37100005, 27.78700066],
       [ 5.15700006,  1.74100006,  1.27900004, ...,  3.77699995,
         1.21800005, 27.78700066],
       ...,
       [ 2.81299996,  3.81800008, -0.81699997, ...,  3.71600008,
         1.523     , 10.27099991],
       [ 9.24300003,  3.81800008,  5.47200012, ...,  3.65499997,
         1.43200004,  9.77799988],
       [10.11400032,  3.54999995,  6.18300009, ...,  3.71600008,
         1.46200001,  9.56700039]]),)
	index=DatetimeIndex(['2016-07-01 00:00:00', '2016-07-01 01:00:00',
               '2016-07-01 02:00:00', '2016-07-01 03:00:00',
               '2016-07-01 04:00:00', '2016-07-01 05:00:00',
               '2016-07-01 06:00:00', '2016-07-01 07:00:00',
               '2016-07-01 08:00:00', '2016-07-01 09:00:00',
               ...
               '2018-06-26 10:00:00', '

tensor([[ 5.8270,  2.0090,  1.5990,  ...,  4.2030,  1.3400, 30.5310],
        [ 5.6930,  2.0760,  1.4920,  ...,  4.1420,  1.3710, 27.7870],
        [ 5.1570,  1.7410,  1.2790,  ...,  3.7770,  1.2180, 27.7870],
        ...,
        [ 2.8130,  3.8180, -0.8170,  ...,  3.7160,  1.5230, 10.2710],
        [ 9.2430,  3.8180,  5.4720,  ...,  3.6550,  1.4320,  9.7780],
        [10.1140,  3.5500,  6.1830,  ...,  3.7160,  1.4620,  9.5670]])

In [28]:
T.index

DatetimeIndex(['2016-07-01 00:00:00', '2016-07-01 01:00:00',
               '2016-07-01 02:00:00', '2016-07-01 03:00:00',
               '2016-07-01 04:00:00', '2016-07-01 05:00:00',
               '2016-07-01 06:00:00', '2016-07-01 07:00:00',
               '2016-07-01 08:00:00', '2016-07-01 09:00:00',
               ...
               '2018-06-26 10:00:00', '2018-06-26 11:00:00',
               '2018-06-26 12:00:00', '2018-06-26 13:00:00',
               '2018-06-26 14:00:00', '2018-06-26 15:00:00',
               '2018-06-26 16:00:00', '2018-06-26 17:00:00',
               '2018-06-26 18:00:00', '2018-06-26 19:00:00'],
              dtype='datetime64[ns]', name='date', length=17420, freq=None)

In [8]:
a
torch.tensor(ds.dataset.values)

tensor([[ 5.8270,  2.0090,  1.5990,  ...,  4.2030,  1.3400, 30.5310],
        [ 5.6930,  2.0760,  1.4920,  ...,  4.1420,  1.3710, 27.7870],
        [ 5.1570,  1.7410,  1.2790,  ...,  3.7770,  1.2180, 27.7870],
        ...,
        [ 2.8130,  3.8180, -0.8170,  ...,  3.7160,  1.5230, 10.2710],
        [ 9.2430,  3.8180,  5.4720,  ...,  3.6550,  1.4320,  9.7780],
        [10.1140,  3.5500,  6.1830,  ...,  3.7160,  1.4620,  9.5670]],
       dtype=torch.float64)

In [5]:
repr(ds)

'                       HUFL   HULL   MUFL   MULL   LUFL   LULL         OT\ndate                                                                     \n2016-07-01 00:00:00   5.827  2.009  1.599  0.462  4.203  1.340  30.531000\n2016-07-01 01:00:00   5.693  2.076  1.492  0.426  4.142  1.371  27.787001\n2016-07-01 02:00:00   5.157  1.741  1.279  0.355  3.777  1.218  27.787001\n2016-07-01 03:00:00   5.090  1.942  1.279  0.391  3.807  1.279  25.044001\n2016-07-01 04:00:00   5.358  1.942  1.492  0.462  3.868  1.279  21.948000\n...                     ...    ...    ...    ...    ...    ...        ...\n2018-06-26 15:00:00  -1.674  3.550 -5.615  2.132  3.472  1.523  10.904000\n2018-06-26 16:00:00  -5.492  4.287 -9.132  2.274  3.533  1.675  11.044000\n2018-06-26 17:00:00   2.813  3.818 -0.817  2.097  3.716  1.523  10.271000\n2018-06-26 18:00:00   9.243  3.818  5.472  2.097  3.655  1.432   9.778000\n2018-06-26 19:00:00  10.114  3.550  6.183  1.564  3.716  1.462   9.567000\n\n[17420 rows x 7 column